# 3. Combine Session Regression Outputs Across ROIs

This notebook combines the session-wise regression outputs generated in the previous notebook into unified visual-region datasets.

The resulting files are used by all subsequent representational drift analyses.

# Notebook Position in the Pipeline

Pipeline order:

`1. nsd_prf_sampling_from_pyramid.ipynb`

`2. nsd_regression_prf_split.ipynb`

`3. Combine_Session__ROIs.ipynb` ← current notebook

`4. save_perms_expand.ipynb`


# Purpose of This Notebook

The previous notebook fits encoding models separately for each visual-region subdivision.

Examples:

- V1v
- V1d
- V2v
- V2d
- V3v
- V3d

For the drift analyses we are interested in full visual regions:

- V1
- V2
- V3
- hV4

This notebook combines the ROI subdivisions into a single ROI representation for each visual region.

It also computes and stores a mean session split that is used throughout the remainder of the pipeline.

# Inputs

This notebook requires the regression outputs generated by:

`2_nsd_regression_prf_split.ipynb`

Input files:

regressPrfSplit_session_v{visualRegion}_sub{subject}{normalization}.pkl

Examples:

- regressPrfSplit_session_v1_sub1_equalStd.pkl
- regressPrfSplit_session_v2_sub1_equalStd.pkl
- regressPrfSplit_session_v3_sub1_equalStd.pkl
- regressPrfSplit_session_v4_sub1_equalStd.pkl

These files contain:

- Session beta responses
- Session standard deviations
- Regression coefficients
- Cross-session prediction metrics
- pRF assignments
- Model goodness-of-fit measures

# Main Operations Performed

For each visual region:

1. Load session-wise regression results.

2. Merge ROI subdivisions
   - V1v + V1d
   - V2v + V2d
   - V3v + V3d

3. Concatenate voxel-level information.

4. Concatenate pRF parameters.

5. Append a "mean split" across sessions.

6. Store all outputs in a single combined structure.

7. Save one file per subject.

# Mean Split

The original regression files contain one entry per session.

This notebook appends an additional split corresponding to the mean across sessions.

Example:

Original:

Split 1
Split 2
...
Split N

After combination:

Split 1
Split 2
...
Split N
Mean Split

Therefore:

nsplits_saved = nsessions + 1



# Outputs

Output file:

regressSessCombineROI_sub{subject}{normalization}.pkl

Examples:

- regressSessCombineROI_sub1_equalStd.pkl
- regressSessCombineROI_sub2_equalStd.pkl

Stored variables include:

- allRoiPrf
- roiNsdCorr
- roiNsdOriCorr
- roiNsdR2
- roiNsdOriR2
- roiNsdRss
- roiNsdOriRss
- voxCoef
- voxOriCoef
- voxConstCoef
- voxConstOriCoef
- sessBetas
- sessStdBetas
- nsplits

# Run code

In [ ]:
import os
import pickle
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


# ============================================================
# CONFIGURATION
# ============================================================
SAVE_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand"


def get_zscore_suffix(to_zscore: int) -> str:
    if to_zscore == 1:
        return "_zscore"
    elif to_zscore == 2:
        return "_zeroMean"
    elif to_zscore == 3:
        return "_equalStd"
    elif to_zscore == 4:
        return "_zeroROImean"
    else:
        return ""


# ============================================================
# HELPER FUNCTION: APPEND MEAN SESSION
# ============================================================
# The original regression files contain one entry per session.
# Later drift analyses expect an additional split corresponding
# to the mean across sessions (matching the MATLAB pipeline).
#
# Input:
#     [sessions x ...]
#
# Output:
#     [sessions + mean x ...]
#
def _append_mean_over_splits(arr: np.ndarray, split_axis: int = 0) -> np.ndarray:
    """
    Append an extra 'mean split' along split_axis (MATLAB behavior: nsplits includes mean).
    Uses nanmean for robustness (matches your pipeline notebook helper).
    """
    if arr is None:
        return None
    arr = np.asarray(arr)
    if arr.shape[split_axis] == 0:
        return arr
    mean_slice = np.nanmean(arr, axis=split_axis, keepdims=True)
    return np.concatenate([arr, mean_slice], axis=split_axis)


# ============================================================
# HELPER FUNCTION: COMBINE ROI SUBDIVISIONS
# ============================================================
# Merges ventral and dorsal subdivisions of the same visual
# region into a single ROI.
#
# Examples:
#     V1v + V1d -> V1
#     V2v + V2d -> V2
#     V3v + V3d -> V3
#
# Voxel-dependent variables are concatenated across voxels.
# pRF parameters are concatenated accordingly so that voxel
# ordering remains consistent.
def combine_rois(nsd: dict, roiPrf: dict, rois: list[int]) -> tuple[dict, dict]:
    """
    Combine multiple ROIs (e.g., V1v + V1d) into one ROI by concatenating voxels.
    Matches MATLAB logic: cat(2, ...) for voxel-dimension arrays; cat(1, ...) for voxel-vectors.
    """

    axis1_fields = [
        "sessBetas",
        "sessStdBetas",
        "pearsonRori",
        "pearsonR",
        "r2split",
        "r2oriSplit",
        "rssSplit",
        "rssOriSplit",
        "voxCoef",
        "voxOriCoef",
        "voxPredOriCoef",
        "voxOriPredOriCoef",
        "voxResidOriCoef",
        "voxOriResidOriCoef",
        "voxPredOriR2",
        "voxOriPredOriR2",
        "voxResidOriR2",
        "voxOriResidOriR2",
        "r2",
        "r2ori",
    ]

    axis0_fields = [
        "roiInd",
        "voxOriCoefCorrWithConst",
        "voxCoefCorrWithConst",
        "voxOriCoefCorr",
        "voxCoefCorr",
    ]

    nsd_combined = {}

    # cat(2, ...) in MATLAB  -> axis=1 in numpy
    for fname in axis1_fields:
        if fname not in nsd:
            continue
        arrs = [nsd[fname][roi] for roi in rois]
        if len(arrs) == 0 or arrs[0] is None:
            nsd_combined[fname] = None
        else:
            nsd_combined[fname] = np.concatenate(arrs, axis=1)

    # cat(1, ...) in MATLAB  -> axis=0 in numpy
    for fname in axis0_fields:
        if fname not in nsd:
            continue
        arrs = [nsd[fname][roi] for roi in rois]
        if len(arrs) == 0 or arrs[0] is None:
            nsd_combined[fname] = None
        else:
            nsd_combined[fname] = np.concatenate(arrs, axis=0)

    # Combine pRF parameters (voxel-vectors)
    first_roi = rois[0]
    prf_keys = list(roiPrf[first_roi].keys())
    roiPrf_combined = {k: None for k in prf_keys}

    for roi in rois:
        for key in prf_keys:
            arr = np.asarray(roiPrf[roi][key])
            roiPrf_combined[key] = arr if roiPrf_combined[key] is None else np.concatenate(
                [roiPrf_combined[key], arr], axis=0
            )

    return nsd_combined, roiPrf_combined

# ============================================================
# MAIN PIPELINE STEP
# ============================================================
# Loads regression results generated by:
#
#     2_nsd_regression_prf_split.ipynb
#
# For each visual region:
#
#     1. Load regression outputs
#     2. Combine ROI subdivisions
#     3. Append mean split
#     4. Store region-level variables
#     5. Save a subject-level combined file
#
# Output:
#
#     regressSessCombineROI_subX.pkl
def regress_session_combine_roi_expand(
    isub: int,
    numregions: int = 4,   # IMPORTANT: set to 4 for V1–V4
    to_zscore: int = 0,
    save_dir: str = SAVE_DIR,
):
    """
    Python version of regressSessionCombineRoi_expand.m
    - Loads regressPrfSplit_session_v{region}_sub{isub}{z}.pkl
    - Combines ventral+dorsal within each region
    - Appends mean split so nsplits includes mean (nsessions = nsplits-1 downstream)
    - Saves regressSessCombineROI_sub{isub}{z}.pkl
    """

    save_dir = os.path.abspath(save_dir)
    zscore_str = get_zscore_suffix(to_zscore)

    print(f"\n=== regress_session_combine_roi_expand: sub={isub}, numregions={numregions}, to_zscore={to_zscore} ===")
    print("Using save_dir:", save_dir)

    allRoiPrf = []
    roiInd_list = []
    roiNsdCorr = []
    roiNsdOriCorr = []
    roiNsdOriR2 = []
    roiNsdR2 = []
    roiNsdOriRss = []
    roiNsdRss = []
    roiNsdR2within = []
    roiNsdOriR2within = []

    voxConstCoef = []
    voxConstOriCoef = []
    voxOriCoefCorrWithConst_list = []
    voxCoefCorrWithConst_list = []
    voxOriCoefCorr_list = []
    voxCoefCorr_list = []
    sessBetas_list = []
    sessStdBetas_list = []

    roiNsdOriPredOriR2 = []
    roiNsdOriResidOriR2 = []
    roiNsdPredOriR2 = []
    roiNsdResidOriR2 = []

    voxOriCoef_list = []
    voxCoef_list = []

    visRoiData = None
    roiNames = None
    combinedRoiNames = None
    prefAnalysis = None

    final_nsplits = None


    # PROCESS EACH VISUAL REGION SEPARATELY
    for visualRegion in range(1, numregions + 1):
        in_name = f"regressPrfSplit_session_v{visualRegion}_sub{isub}{zscore_str}.pkl"
        in_path = os.path.join(save_dir, in_name)
        if not os.path.exists(in_path):
            raise FileNotFoundError(f"Missing input file: {in_path}")

        print(f"\n-- Loading {in_path}")
        # Load session-wise regression outputs
        with open(in_path, "rb") as f_in:
            data = pickle.load(f_in)

        nsd = data["nsd"]           # dict: field -> dict(roinum -> array)
        rois = data["rois"]         # e.g. [0,1] or [2,3] etc (indices)
        roiPrf = data["roiPrf"]     # dict: roinum -> dict(prfparam -> vec)
        nsplits = data["nsplits"]   # original session count (no mean yet)
        roi_names = data.get("roi_names", None)

        if roiNames is None:
            roiNames = roi_names

        # Combine ventral+dorsal into one ROI (even if only one ROI exists)
        nsd_combined, roiPrf_combined = combine_rois(nsd, roiPrf, rois)

        # Append mean split to EVERYTHING that is split-indexed (axis 0 = splits/sessions)
        split_keys = [
            "sessBetas", "sessStdBetas",
            "voxOriCoefCorrWithConst", "voxCoefCorrWithConst",
            "voxOriCoefCorr", "voxCoefCorr",
            "voxCoef", "voxOriCoef",
            "voxPredOriCoef", "voxOriPredOriCoef",
            "voxResidOriCoef", "voxOriResidOriCoef",
            "pearsonRori", "pearsonR",
            "r2", "r2ori",
            "r2split", "r2oriSplit",
            "rssSplit", "rssOriSplit",
            "voxPredOriR2", "voxOriPredOriR2",
            "voxResidOriR2", "voxOriResidOriR2",
        ]
        for key in split_keys:
            if key in nsd_combined and nsd_combined[key] is not None:
                nsd_combined[key] = _append_mean_over_splits(nsd_combined[key], split_axis=0)

        nsplits_with_mean = nsplits + 1

        # Store region outputs (one entry per visualRegion)
        allRoiPrf.append(roiPrf_combined)
        roiInd_list.append(nsd_combined["roiInd"])

        roiNsdCorr.append(nsd_combined["pearsonR"])
        roiNsdOriCorr.append(nsd_combined["pearsonRori"])
        roiNsdOriR2.append(nsd_combined["r2oriSplit"])
        roiNsdR2.append(nsd_combined["r2split"])
        roiNsdOriRss.append(nsd_combined["rssOriSplit"])
        roiNsdRss.append(nsd_combined["rssSplit"])

        roiNsdR2within.append(nsd_combined["r2"])
        roiNsdOriR2within.append(nsd_combined["r2ori"])

        # Constant coefficient = last predictor
        voxCoef_arr = nsd_combined["voxCoef"]
        voxOriCoef_arr = nsd_combined["voxOriCoef"]
        voxConstCoef.append(voxCoef_arr[:, :, -1])
        voxConstOriCoef.append(voxOriCoef_arr[:, :, -1])

        voxOriCoefCorrWithConst_list.append(nsd_combined["voxOriCoefCorrWithConst"])
        voxCoefCorrWithConst_list.append(nsd_combined["voxCoefCorrWithConst"])
        voxOriCoefCorr_list.append(nsd_combined["voxOriCoefCorr"])
        voxCoefCorr_list.append(nsd_combined["voxCoefCorr"])

        sessBetas_list.append(nsd_combined["sessBetas"])
        sessStdBetas_list.append(nsd_combined["sessStdBetas"])

        roiNsdOriPredOriR2.append(nsd_combined["voxOriPredOriR2"])
        roiNsdOriResidOriR2.append(nsd_combined["voxOriResidOriR2"])
        roiNsdPredOriR2.append(nsd_combined["voxPredOriR2"])
        roiNsdResidOriR2.append(nsd_combined["voxResidOriR2"])

        voxOriCoef_list.append(voxOriCoef_arr)
        voxCoef_list.append(voxCoef_arr)

        if final_nsplits is None:
            final_nsplits = nsplits_with_mean
        else:
            assert final_nsplits == nsplits_with_mean, "nsplits mismatch across regions!"

        print(f"Region v{visualRegion}: combined voxels = {voxCoef_arr.shape[1]}, nsplits(with mean)={nsplits_with_mean}")

    out_name = f"regressSessCombineROI_sub{isub}{zscore_str}.pkl"
    out_path = os.path.join(save_dir, out_name)

    # ============================================================
    # BUILD FINAL SUBJECT OUTPUT STRUCTURE
    # ============================================================
    # This dictionary contains all combined visual regions (V1–hV4) for a single subject.
    out_dict = {
        "allRoiPrf": allRoiPrf,
        "roiNsdCorr": roiNsdCorr,
        "roiNsdOriCorr": roiNsdOriCorr,
        "roiNsdOriR2": roiNsdOriR2,
        "roiNsdR2": roiNsdR2,
        "roiNsdOriRss": roiNsdOriRss,
        "roiNsdRss": roiNsdRss,
        "numregions": numregions,
        "roiNsdResidOriR2": roiNsdResidOriR2,
        "roiNsdOriResidOriR2": roiNsdOriResidOriR2,
        "roiNsdPredOriR2": roiNsdPredOriR2,
        "roiNsdOriPredOriR2": roiNsdOriPredOriR2,
        "visRoiData": visRoiData,
        "roiNames": roiNames,
        "combinedRoiNames": combinedRoiNames,
        "roiInd": roiInd_list,
        "prefAnalysis": prefAnalysis,
        "nsplits": final_nsplits,                 # IMPORTANT: includes mean
        "roiNsdOriR2within": roiNsdOriR2within,
        "roiNsdR2within": roiNsdR2within,
        "voxOriCoefCorrWithConst": voxOriCoefCorrWithConst_list,
        "voxCoefCorrWithConst": voxCoefCorrWithConst_list,
        "voxOriCoefCorr": voxOriCoefCorr_list,
        "voxCoefCorr": voxCoefCorr_list,
        "sessBetas": sessBetas_list,              # IMPORTANT: includes mean row
        "sessStdBetas": sessStdBetas_list,        # IMPORTANT: includes mean row
        "voxConstCoef": voxConstCoef,
        "voxConstOriCoef": voxConstOriCoef,
        "voxOriCoef": voxOriCoef_list,
        "voxCoef": voxCoef_list,
    }

    print(f"\nSaving combined ROI data to: {out_path}")
    with open(out_path, "wb") as f_out:
        pickle.dump(out_dict, f_out, protocol=pickle.HIGHEST_PROTOCOL)

    print("Done.")

# ============================================================
# RUN COMBINATION FOR ALL NSD SUBJECTS
# ============================================================
# Creates one combined ROI file per subject
# ============================================================
# Run for subject V1–V4:
regress_session_combine_roi_expand(isub=1, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=2, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=3, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=4, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=5, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=6, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=7, numregions=4, to_zscore=3)
regress_session_combine_roi_expand(isub=8, numregions=4, to_zscore=3)


# CHECK OUTPUT FILE
# ============================================================
 This cell loads the combined ROI file created by:
#
     3_Combine_Session__ROIs.ipynb
#
and checks that the saved output has the expected structure.
#
# Main goals:
     1. Confirm that the file exists and can be loaded.
     2. Inspect the top-level keys saved in the output dictionary.
     3. Check metadata such as number of splits and visual regions.
     4. Verify that important fields have the expected shape.
     5. Check for NaNs or unusual values.
#
# This should be run before using the file in downstream drift analyses.

In [ ]:
import os
import pickle
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
SUB = 6

path = os.path.join(SAVE_DIR, f"regressSessCombineROI_sub{SUB}.pkl")

with open(path, "rb") as f:
    D = pickle.load(f)

print("="*80)
print("FILE:", path)
print("="*80)
print("\nTop-level keys:")
for k in D.keys():
    print("  -", k)

print("\nBasic metadata:")
print("  nsplits:", D.get("nsplits"))
print("  numregions:", D.get("numregions"))
print("  roiNames:", D.get("roiNames"))
print()

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def summarize_array(name, arr):
    arr = np.asarray(arr)
    print(f"    shape={arr.shape}, dtype={arr.dtype}")
    if np.issubdtype(arr.dtype, np.number):
        finite = arr[np.isfinite(arr)]
        if finite.size > 0:
            print(f"      min={finite.min():.4g}, max={finite.max():.4g}, "
                  f"mean={finite.mean():.4g}, std={finite.std():.4g}")
            print(f"      NaN fraction={1 - finite.size/arr.size:.4%}")
        else:
            print("      All values are NaN or non-finite.")
    print()

def inspect_field(field_name):
    print("="*80)
    print(f"FIELD: {field_name}")
    print("="*80)

    obj = D.get(field_name)

    if obj is None:
        print("  -> None\n")
        return

    if isinstance(obj, list):
        print(f"  List of length {len(obj)} (one per visual region)")
        for i, item in enumerate(obj):
            print(f"  Region {i+1}:")
            if isinstance(item, dict):
                print(f"    dict with keys: {list(item.keys())}")
            elif isinstance(item, np.ndarray):
                summarize_array(f"{field_name}[{i}]", item)
            else:
                print(f"    type={type(item)}\n")
        return

    if isinstance(obj, dict):
        print("  Dict with keys:")
        for k, v in obj.items():
            print("   ", k, "->", type(v))
        print()
        return

    if isinstance(obj, np.ndarray):
        summarize_array(field_name, obj)
        return

    print("  type:", type(obj))
    print()

# ------------------------------------------------------------
# Inspect important fields
# ------------------------------------------------------------

fields_to_check = [
    "roiNsdCorr",
    "roiNsdOriCorr",
    "roiNsdR2",
    "roiNsdOriR2",
    "roiNsdR2within",
    "roiNsdOriR2within",
    "voxCoef",
    "voxOriCoef",
    "voxConstCoef",
    "sessBetas",
    "sessStdBetas",
    "roiInd",
    "allRoiPrf",
]

for field in fields_to_check:
    if field in D:
        inspect_field(field)


# DIAGNOSTIC CHECK: NUMERIC SANITY PER VISUAL REGION
# ============================================================
# This cell checks the main prediction-quality outputs saved in the combined ROI file.
#
# For each visual region, it summarizes:
     - roiNsdCorr: cross-session Pearson correlation
     - roiNsdR2: cross-session R² / CVR² values
#
 The final split is excluded using [:-1] because the last split
 is the appended mean split, not an original session.
#
# Use this cell to check for:
     - unexpected NaN values
     - extremely large or small values
     - unusual distributions
     - shape mismatches across regions


In [ ]:
for roi in range(D["numregions"]):
    print("\n" + "="*60)
    print(f"Region {roi+1} numeric sanity")
    print("="*60)

    r = D["roiNsdCorr"][roi][:-1,:,:]
    r2 = D["roiNsdR2"][roi][:-1,:,:]

    print("Pearson:")
    summarize_array("pearson", r)

    print("R2:")
    summarize_array("r2", r2)


# OPTIONAL / LEGACY CHECK: PEARSON r VS CVR²
# ============================================================
This diagnostic compares Pearson r and CVR² for one selected
high-performing voxel.

# Goal:
     Check whether the Pearson correlation and CVR² values are
     numerically reasonable for the same voxel, split, and lag.

# Important:
     Pearson r² is NOT expected to be identical to CVR².

 Pearson r² measures squared correlation between prediction and
 observed response.

 CVR² measures cross-validated explained variance and depends on
 prediction error relative to the variance of the true response.

 Therefore, this cell is only a sanity check, not an equality test.

# Note:
     This cell uses an older .npz path/version of the combined file.
     If the current pipeline saves .pkl files, update the loading code
     before using this check.


In [ ]:
# pearson check
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Load the combined output file
path = "/content/drive/My Drive/V1_Drift/repDrift_expand/v1/Subject 3/regressSessCombineROI_sub3.npz"
D = np.load(path, allow_pickle=True)

# Select a good voxel based on within-session R²
roi_r2 = D["v1_roi_r2"]   # shape: (n_vox,)
ivox = int(np.argmax(roi_r2))
print(f"Chosen good voxel index: {ivox}, mean within-session R² = {roi_r2[ivox]:.4f}")

# Extract CVR² and Pearson matrices
roiNsdCorr_dict = D["roiNsdCorr"].item()   #  dict: key=1 → visual region 1
roiNsdR2_dict   = D["roiNsdR2"].item()

corr = roiNsdCorr_dict[1]   # shape: (nsplits, nvox, nlags)
cvr2 = roiNsdR2_dict[1]     # shape: (nsplits, nvox, nlags)

nsplits, nvox, nlags = corr.shape
print(f"corr shape = {corr.shape}, cvr2 shape = {cvr2.shape}")

# Select the mean split and zero-lag index
mean_split_idx = nsplits - 1

# choose index for lag=0
zero_lag_idx = 15
r_val   = float(corr[mean_split_idx, ivox, zero_lag_idx])
cvr2_val = float(cvr2[mean_split_idx, ivox, zero_lag_idx])

print(f"\nVoxel {ivox}, mean split, zero-lag index {zero_lag_idx}:")
print(f"  Pearson r   = {r_val:.4f}")
print(f"  r^2         = {r_val**2:.4f}")
print(f"  CVR² (from file) = {cvr2_val:.4f}")


# OPTIONAL DIAGNOSTIC CHECK: ORIENTATION MODEL SCALE
# ===========================================================
This cell compares the scale of the non-orientation and orientation-based model components.
#
# It checks:
#     1. Regression coefficient magnitudes
#     2. pRF-sampled feature magnitudes
#     3. Prediction magnitudes for a small sample of images/voxels

 This is useful when debugging cases where the orientation model
 gives unexpectedly large/small predictions, unstable coefficients,
 or unusual cross-session performance.

 Inputs used:
     - Combined regression output:
         regressSessCombineROI_sub{subject}{normalization}.pkl

     - pRF sampling output:
         prfSampleStim_v{visualRegion}_sub{subject}.h5

This cell is diagnostic only and does not save new pipeline outputs.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import numpy as np
import h5py
import pickle

BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
PRF_FOLDER  = "/content/drive/MyDrive/V1_Drift/prfsample_expand"

def _zstr(to_zscore):
    return {
        0: "",
        1: "_zscore",
        2: "_zeroMean",
        3: "_equalStd",
        4: "_zeroROImean",
    }.get(to_zscore, "")


# Main diagnostic function for comparing feature, coefficient,
# and prediction scales between non-orientation and orientation models.
def check_ori_scale(isub=1, vreg=1, to_zscore=0):
    import os, pickle, h5py
    import numpy as np

    z = _zstr(to_zscore)

    with open(os.path.join(BASE_FOLDER, f"regressSessCombineROI_sub{isub}{z}.pkl"), "rb") as f:
        D = pickle.load(f)

    vox_coef = np.asarray(D["voxCoef"][vreg - 1])
    vox_ori_coef = np.asarray(D["voxOriCoef"][vreg - 1])

    prf_path = os.path.join(PRF_FOLDER, f"prfSampleStim_v{vreg}_sub{isub}.h5")
    with h5py.File(prf_path, "r") as f:
        lev_keys = sorted(list(f["prfSampleLev"].keys()))
        ori_keys = sorted(list(f["prfSampleLevOri"].keys()))

        if vreg < 4:
            lev = np.concatenate([f[f"prfSampleLev/{k}"][:] for k in lev_keys], axis=1)
            levO = np.concatenate([f[f"prfSampleLevOri/{k}"][:] for k in ori_keys], axis=1)
        else:
            lev = f[f"prfSampleLev/{lev_keys[0]}"][:]
            levO = f[f"prfSampleLevOri/{ori_keys[0]}"][:]

    print("=== Coefficient scale ===")
    print("vox_coef abs mean:     ", np.nanmean(np.abs(vox_coef)))
    print("vox_ori_coef abs mean: ", np.nanmean(np.abs(vox_ori_coef)))
    print("vox_coef abs max:      ", np.nanmax(np.abs(vox_coef)))
    print("vox_ori_coef abs max:  ", np.nanmax(np.abs(vox_ori_coef)))

    print("\n=== Feature scale ===")
    print("prfSampleLev abs mean:    ", np.nanmean(np.abs(lev)))
    print("prfSampleLevOri abs mean: ", np.nanmean(np.abs(levO)))
    print("prfSampleLev abs max:     ", np.nanmax(np.abs(lev)))
    print("prfSampleLevOri abs max:  ", np.nanmax(np.abs(levO)))

    print("\n=== Prediction scale, small sample ===")
    # check first 10 voxels, first session, first 100 images
    nimgs = 100
    num_levels = levO.shape[2]
    num_orientations = levO.shape[3]

    pred_nonori_all = []
    pred_ori_all = []

    for ivox in range(min(10, lev.shape[1])):
        X_lev = np.concatenate([
            lev[:nimgs, ivox, :],
            np.ones((nimgs, 1))
        ], axis=1)

        X_ori_flat = levO[:nimgs, ivox, :, :].reshape(
            nimgs,
            num_levels * num_orientations,
            order="F"
        )

        X_ori = np.concatenate([
            X_ori_flat,
            lev[:nimgs, ivox, num_levels:num_levels + 2],
            np.ones((nimgs, 1))
        ], axis=1)

        pred_nonori_all.append(X_lev @ vox_coef[0, ivox, :])
        pred_ori_all.append(X_ori @ vox_ori_coef[0, ivox, :])

    pred_nonori_all = np.asarray(pred_nonori_all)
    pred_ori_all = np.asarray(pred_ori_all)

    print("nonori pred mean:", np.nanmean(pred_nonori_all))
    print("ori pred mean:   ", np.nanmean(pred_ori_all))
    print("nonori pred std: ", np.nanstd(pred_nonori_all))
    print("ori pred std:    ", np.nanstd(pred_ori_all))
    print("nonori pred min/max:", np.nanmin(pred_nonori_all), np.nanmax(pred_nonori_all))
    print("ori pred min/max:   ", np.nanmin(pred_ori_all), np.nanmax(pred_ori_all))

In [ ]:
check_ori_scale(isub=1, vreg=1, to_zscore=0)